# 6장. 분류 평가 지표

정확도가 높으면 좋은 모델인가.

**준비:** 경기데이터드림(data.gg.go.kr)에서 받은 「일반음식점(중국식) 현황」 CSV를 왼쪽 파일 창에 올린 뒤 위에서부터 차례로 실행한다. 『데이터 분석 완전 정복』 6장에서 쓴 파일 그대로다.

## 6.1 정확도와 기준율

In [ ]:
import glob
import numpy as np
import pandas as pd

pd.set_option("display.unicode.east_asian_width", True)
pd.set_option("display.width", 200)

def first_line(path):
    with open(path, encoding="cp949") as f:
        return f.readline()

name = [f for f in glob.glob("*.csv") if first_line(f).startswith("시군명,사업장명")][0]
raw = pd.read_csv(name, encoding="cp949", dtype=str)

raw["시군"] = raw["시군명"].str.strip()
raw["구분"] = np.where(raw["시군명"].str.endswith(" "), "공백 있음", "공백 없음")
shops = raw.sort_values(["구분", "영업상태명"], ascending=[True, False]).drop_duplicates(["시군", "사업장명", "인허가일자"]).copy()

shops["개업"] = pd.to_datetime(shops["인허가일자"], format="%Y%m%d", errors="coerce")
shops["폐업"] = pd.to_datetime(shops["폐업일자"], format="%Y%m%d", errors="coerce")
shops = shops.dropna(subset=["개업"]).reset_index(drop=True)
END = max(shops["개업"].max(), shops["폐업"].max())
shops["끝"] = shops["폐업"].fillna(END + pd.Timedelta(days=1))
shops["위도"] = pd.to_numeric(shops["WGS84위도"], errors="coerce")
shops["경도"] = pd.to_numeric(shops["WGS84경도"], errors="coerce")
shops["지번"] = shops["소재지지번주소"].str.replace(r"\s+", " ", regex=True).str.extract(r"^(.*?\d+(?:-\d+)?)")[0]
print(len(raw), len(shops), END.date())

In [ ]:
def snapshot(day):
    """day에 영업 중이던 가게마다 한 줄. 특성은 그날 알 수 있는 값으로만 만들고, 그 뒤 3년 안에 폐업했는지를 레이블로 붙인다."""
    day = pd.Timestamp(day)
    began, ended = shops["개업"].to_numpy(), shops["끝"].to_numpy()
    open_now = (began < day.to_datetime64()) & (ended > day.to_datetime64())
    gone = shops["폐업"].notna().to_numpy() & (ended < day.to_datetime64())
    lately = gone & (ended >= (day - pd.DateOffset(years=3)).to_datetime64())

    a = shops[open_now].copy()
    a["기준일"] = day
    a["업력"] = (day - a["개업"]).dt.days / 365.25
    a["지번폐업"] = a["지번"].map(shops[gone].groupby("지번").size()).fillna(0)      # 같은 지번에서 먼저 폐업한 중국집 수

    def within(mask, metres):
        """a의 가게마다, mask에 해당하는 가게 가운데 metres 안에 있는 것의 수."""
        lat, lon = shops.loc[mask, "위도"].to_numpy(), shops.loc[mask, "경도"].to_numpy()
        dy = (a["위도"].to_numpy()[:, None] - lat[None, :]) * 111195
        dx = (a["경도"].to_numpy()[:, None] - lon[None, :]) * 111195 * np.cos(np.radians(a["위도"].to_numpy()))[:, None]
        return np.where(a["위도"].isna(), np.nan, (dx ** 2 + dy ** 2 < metres ** 2).sum(axis=1))

    a["주변"] = within(open_now, 500) - 1            # 반경 500m 안에서 영업 중인 다른 중국집
    a["주변폐업"] = within(lately, 1000)              # 반경 1km 안에서 지난 3년 사이 폐업한 중국집
    a["폐업함"] = (a["폐업"].notna() & (a["폐업"] <= day + pd.DateOffset(years=3))).astype(int)
    return a

train = pd.concat([snapshot(d) for d in ["2011-04-03", "2014-04-03", "2017-04-03", "2020-04-03"]], ignore_index=True)
test = snapshot("2023-04-03").reset_index(drop=True)
y = test["폐업함"].to_numpy()

print(train.groupby("기준일")["폐업함"].agg(["size", "mean"]).round(3).to_string())
print(len(test), int(y.sum()), round(float(y.mean()), 3))

In [ ]:
def accuracy(said, actual):
    return round(float((said == actual).mean() * 100), 1)

print(accuracy(np.zeros(len(y), dtype=int), y))

In [ ]:
SIGUN = sorted(shops["시군"].unique())
YEARS = [0, 1, 2, 3, 5, 10, 20]

def onehot(s, cats):
    return (np.asarray(s)[:, None] == np.asarray(cats)[None, :]).astype(float)

def columns(df):
    """업력 구간 가변수 7개, 시군 가변수 30개, 같은 지번의 폐업 이력, 주변의 영업 중·최근 폐업 가게 수(로그), 좌표 없음 표시."""
    return np.column_stack([
        onehot(np.digitize(df["업력"], YEARS) - 1, range(len(YEARS))),
        onehot(df["시군"], SIGUN)[:, 1:],
        np.minimum(df["지번폐업"], 3),
        np.log1p(df["주변"].fillna(0)), np.log1p(df["주변폐업"].fillna(0)), df["주변"].isna().astype(float),
    ])

coef = np.linalg.lstsq(columns(train), train["폐업함"].to_numpy(), rcond=None)[0]
score = columns(test) @ coef
print(columns(train).shape, np.quantile(score, [0, 0.1, 0.5, 0.9, 1]).round(3).tolist())
print(coef[:7].round(3).tolist())

In [ ]:
said = (score >= 0.5).astype(int)
print(accuracy(said, y), int(said.sum()), int((said & y).sum()))

In [ ]:
!pip -q install koreanize-matplotlib

In [ ]:
import matplotlib.pyplot as plt
import koreanize_matplotlib
from matplotlib.colors import to_rgba

NAVY, BRICK, TEAL, GRAY = "#16324f", "#b5532a", "#2a7f73", "#9aa1ab"
BOX = {"fc": "white", "ec": "none", "alpha": 0.9, "pad": 1.5}


def frame(ax, title=None, color=NAVY, grid="y"):
    """그래프마다 반복하는 꾸밈: 제목, 보조선, 테두리 정리."""
    if title:
        ax.set_title(title, fontsize=9, loc="left", color=color, weight="bold")
    ax.grid(axis=grid, color="#e3e7ec", lw=0.6)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=8)
    ax.spines[["top", "right"]].set_visible(False)

In [ ]:
# 그림 6-2
bins = np.arange(0.08, 0.53, 0.01)
fig, ax = plt.subplots(figsize=(7, 3.9))
ax.hist(score[y == 0], bins=bins, density=True, color=to_rgba(NAVY, 0.22), edgecolor=NAVY, lw=0.6, label=f"영업을 유지한 가게 {int((y == 0).sum()):,}곳")
ax.hist(score[y == 1], bins=bins, density=True, color=to_rgba(BRICK, 0.45), edgecolor=BRICK, lw=0.6, label=f"3년 안에 폐업한 가게 {int(y.sum()):,}곳")
top = ax.get_ylim()[1]
ax.set_ylim(0, top * 1.22)
for cut, text, xoff in [(0.5, "임곗값 0.5\n이 선의 오른쪽에 2곳", -0.008), (0.2, "임곗값 0.2", 0.006)]:
    ax.axvline(cut, color=TEAL, lw=1.2, ls=(0, (4, 3)))
    ax.text(cut + xoff, top * 1.08, text, fontsize=8, color=TEAL, ha="right" if xoff < 0 else "left", va="center", linespacing=1.4)
ax.set_xlabel("모델의 점수", fontsize=8)
ax.set_ylabel("밀도(막대의 넓이의 합이 집단마다 1)", fontsize=8)
ax.legend(frameon=False, fontsize=8, loc="center right", bbox_to_anchor=(0.9, 0.5))
frame(ax)
fig.tight_layout()
fig.savefig("fig_6_2.png", dpi=300)
plt.show()

## 6.2 혼동 행렬, 정밀도, 재현율

In [ ]:
def four(said, actual):
    """참 양성(TP), 거짓 양성(FP), 거짓 음성(FN), 참 음성(TN)의 수."""
    return (int((said & actual).sum()), int((said & (1 - actual)).sum()),
            int(((1 - said) & actual).sum()), int(((1 - said) & (1 - actual)).sum()))

def report(cut):
    said = (score >= cut).astype(int)
    tp, fp, fn, tn = four(said, y)
    precision, recall = tp / max(tp + fp, 1), tp / (tp + fn)
    return {"임곗값": cut, "폐업 예측": tp + fp, "TP": tp, "FP": fp, "FN": fn,
            "정확도": round((tp + tn) / len(y) * 100, 1),
            "정밀도": round(precision * 100, 1), "재현율": round(recall * 100, 1),
            "F1": round(2 * precision * recall / max(precision + recall, 1e-12), 3)}

said = (score >= 0.2).astype(int)
tp, fp, fn, tn = four(said, y)
print(pd.DataFrame([[tp, fn], [fp, tn]], index=["실제: 폐업", "실제: 유지"], columns=["예측: 폐업", "예측: 유지"]).to_string())

In [ ]:
CUTS = [0.5, 0.4, 0.3, 0.25, 0.2, 0.15, 0.1, 0.0]
sweep = pd.DataFrame([report(c) for c in CUTS]).set_index("임곗값")
print(sweep.to_string())

In [ ]:
# 그림 6-4
fine = np.round(np.arange(0.09, 0.42, 0.005), 3)
curve = pd.DataFrame([report(c) for c in fine]).set_index("임곗값")
fig, ax = plt.subplots(figsize=(7, 4.0))
ax.plot(curve.index, curve["정확도"], color=GRAY, lw=1.8, label="정확도")
ax.plot(curve.index, curve["재현율"], color=BRICK, lw=1.8, label="재현율")
ax.plot(curve.index, curve["정밀도"], color=NAVY, lw=1.8, label="정밀도")
ax.axhline(y.mean() * 100, color=NAVY, lw=0.8, ls=(0, (2, 2)))
ax.text(0.412, y.mean() * 100 - 4.2, f"기준율 {y.mean() * 100:.1f}%", fontsize=8, color=NAVY, ha="right")
ax.axvline(0.2, color=TEAL, lw=1.0, ls=(0, (4, 3)))
r = report(0.2)
ax.text(0.205, 93, f"임곗값 0.2\n정밀도 {r['정밀도']}%, 재현율 {r['재현율']}%, 정확도 {r['정확도']}%", fontsize=8, color=TEAL, va="center", linespacing=1.4)
ax.set_xlim(0.09, 0.415); ax.set_ylim(0, 104)
ax.set_xlabel("임곗값(점수가 이 값 이상이면 폐업으로 예측)", fontsize=8)
ax.set_ylabel("%", fontsize=8)
ax.legend(frameon=False, fontsize=8, loc="center right")
frame(ax)
fig.tight_layout()
fig.savefig("fig_6_4.png", dpi=300)
plt.show()

## 6.3 임곗값 정하기: 오류의 비용과 기대 이익

In [ ]:
def worth(cut, cost=30, gain=150):
    """폐업으로 예측한 가게를 모두 방문해 지원한다. 방문 1곳에 cost만 원이 들고, 실제로 폐업할 가게였다면 gain만 원의 편익이 생긴다."""
    r = report(cut)
    return r["TP"] * gain - r["폐업 예측"] * cost

print(pd.DataFrame({"편익 150만 원": [worth(c) for c in CUTS], "편익 100만 원": [worth(c, gain=100) for c in CUTS],
                    "편익 300만 원": [worth(c, gain=300) for c in CUTS]}, index=CUTS).to_string())

fine2 = np.round(np.arange(0.09, 0.42, 0.005), 3)
for gain in [100, 150, 300]:
    values = [worth(c, gain=gain) for c in fine2]
    print(gain, round(30 / gain, 2), float(fine2[int(np.argmax(values))]), int(max(values)))

In [ ]:
# 그림 6-5
fig, ax = plt.subplots(figsize=(7, 4.0))
for gain, col in [(300, TEAL), (150, NAVY), (100, BRICK)]:
    values = np.array([worth(c, gain=gain) for c in fine2]) / 10000
    ax.plot(fine2, values, color=col, lw=1.8, label=f"편익 {gain}만 원 (비용 ÷ 편익 = {30 / gain:.1f})")
    k = int(np.argmax(values))
    ax.scatter([fine2[k]], [values[k]], color=col, s=34, zorder=3)
    ax.plot([30 / gain] * 2, [-8.5, -7.3], color=col, lw=2.2)
ax.axhline(0, color=GRAY, lw=0.8)
ax.text(0.405, -6.6, "아래쪽 굵은 눈금: 비용 ÷ 편익", fontsize=7.5, color="#6b7480", ha="right")
ax.text(0.405, -2.6, "가로선(0)보다 위: 지원 사업의 순이익이 양수", fontsize=7.5, color="#6b7480", ha="right")
ax.set_xlim(0.09, 0.41); ax.set_ylim(-8.5, 17.5)
ax.set_xlabel("임곗값", fontsize=8)
ax.set_ylabel("순이익(억 원)", fontsize=8)
ax.legend(frameon=False, fontsize=8, loc="upper right", title="점: 순이익이 가장 큰 임곗값", title_fontsize=8)
frame(ax)
fig.tight_layout()
fig.savefig("fig_6_5.png", dpi=300)
plt.show()

In [ ]:
order = np.argsort(-score)
rows = []
for share in [0.05, 0.1, 0.2, 0.3, 0.5, 1.0]:
    k = int(len(y) * share)
    hit = y[order[:k]]
    rows.append((f"{int(share * 100)}%", k, int(hit.sum()), round(float(hit.mean() * 100), 1),
                 round(float(hit.mean() / y.mean()), 2), round(float(hit.sum() / y.sum() * 100), 1)))
print(pd.DataFrame(rows, columns=["점수 상위", "방문할 곳", "그중 폐업", "정밀도", "향상도", "재현율"]).to_string(index=False))

## 6.4 임곗값 없이 평가하기: ROC 곡선, AUC, 정밀도–재현율 곡선

In [ ]:
def auc(actual, s):
    """폐업한 가게와 유지한 가게를 한 곳씩 짝지을 때 폐업한 쪽의 점수가 더 높은 짝의 비율. 점수가 같으면 0.5로 센다."""
    rank = pd.Series(s).rank().to_numpy()
    n1 = actual.sum()
    return round(float((rank[actual == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * (len(actual) - n1))), 3)

rng = np.random.default_rng(0)
print(auc(y, score), auc(train["폐업함"].to_numpy(), columns(train) @ coef))
print(auc(y, -test["업력"].to_numpy()), auc(y, rng.random(len(y))))

hits = np.cumsum(y[order])
k = np.arange(1, len(y) + 1)
recall, precision = hits / y.sum(), hits / k
false_alarm = (k - hits) / (len(y) - y.sum())
print(round(float(np.trapezoid(np.r_[0, recall], np.r_[0, false_alarm])), 3), round(float((precision * y[order]).sum() / y.sum()), 3))

In [ ]:
# 그림 6-6
fig, ax = plt.subplots(figsize=(5.6, 5.0))
ax.plot(np.r_[0, false_alarm], np.r_[0, recall], color=NAVY, lw=2.0)
ax.plot([0, 1], [0, 1], color=GRAY, lw=0.9, ls=(0, (4, 3)))
ax.fill_between(np.r_[0, false_alarm], np.r_[0, recall], color=to_rgba(NAVY, 0.07))
for cut, dx, dy, ha in [(0.3, 0.03, -0.035, "left"), (0.25, 0.06, -0.075, "left"), (0.2, -0.02, 0.02, "right"), (0.15, -0.02, 0.02, "right")]:
    said_ = score >= cut
    fx, ty = (said_ & (y == 0)).sum() / (y == 0).sum(), (said_ & (y == 1)).sum() / y.sum()
    ax.scatter([fx], [ty], color=BRICK, s=30, zorder=3)
    ax.text(fx + dx, ty + dy, f"임곗값 {cut}", fontsize=8, color=BRICK, ha=ha)
ax.text(0.56, 0.2, f"곡선 아래의 넓이\nAUC = {auc(y, score):.3f}", fontsize=9, color=NAVY, linespacing=1.5)
ax.text(0.8, 0.74, "무작위로 점수를 줄 때", fontsize=7.5, color="#6b7480", rotation=41, ha="center", va="top")
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal")
ax.set_xlabel("거짓 양성률(유지한 가게 가운데 폐업으로 예측한 비율)", fontsize=8)
ax.set_ylabel("재현율(폐업한 가게 가운데 폐업으로 예측한 비율)", fontsize=8)
frame(ax, grid="both")
fig.tight_layout()
fig.savefig("fig_6_6.png", dpi=300)
plt.show()

In [ ]:
# 그림 6-7
fig, ax = plt.subplots(figsize=(7, 3.9))
ax.plot(recall[20:] * 100, precision[20:] * 100, color=NAVY, lw=1.8)
ax.axhline(y.mean() * 100, color=GRAY, lw=0.9, ls=(0, (4, 3)))
ax.text(99, y.mean() * 100 - 2.6, f"기준율 {y.mean() * 100:.1f}% (모든 가게를 폐업으로 예측할 때의 정밀도)", fontsize=8, color="#6b7480", ha="right")
for cut, dx, dy in [(0.3, 1.5, -5.5), (0.25, 1.5, 2.0), (0.2, 1.5, 2.0), (0.15, -1.5, 2.6)]:
    r = report(cut)
    ax.scatter([r["재현율"]], [r["정밀도"]], color=BRICK, s=30, zorder=3)
    ax.text(r["재현율"] + dx, r["정밀도"] + dy, f"임곗값 {cut}", fontsize=8, color=BRICK, ha="left" if dx > 0 else "right")
ax.set_xlim(0, 100); ax.set_ylim(0, 50)
ax.set_xlabel("재현율(%)", fontsize=8)
ax.set_ylabel("정밀도(%)", fontsize=8)
frame(ax)
fig.tight_layout()
fig.savefig("fig_6_7.png", dpi=300)
plt.show()

In [ ]:
keep = (y == 0) | (rng.random(len(y)) < 0.2)          # 폐업한 가게를 5곳 가운데 1곳만 남긴다
y2, s2 = y[keep], score[keep]
o2 = np.argsort(-s2)
top = int(len(y2) * 0.1)
print(round(float(y2.mean() * 100), 1), auc(y2, s2), round(float(y2[o2[:top]].mean() * 100), 1),
      round(float(y2[o2[:top]].mean() / y2.mean()), 2))

def rates(actual, s, cut):
    said_ = s >= cut
    return (said_ & (actual == 1)).sum() / (actual == 1).sum(), (said_ & (actual == 0)).sum() / (actual == 0).sum(), (actual[said_] == 1).mean()

tpr, fpr, prec = rates(y, score, 0.2)
tpr2, fpr2, prec2 = rates(y2, s2, 0.2)
pi2 = y2.mean()
print(round(float(tpr), 3), round(float(fpr), 3), round(float(prec * 100), 1))
print(round(float(tpr2), 3), round(float(fpr2), 3), round(float(prec2 * 100), 1), round(float(pi2 * tpr / (pi2 * tpr + (1 - pi2) * fpr) * 100), 1))

In [ ]:
# 그림 6-8
pi = np.linspace(0.005, 0.5, 200)
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(pi * 100, pi * tpr / (pi * tpr + (1 - pi) * fpr) * 100, color=NAVY, lw=2.0, label=f"임곗값 0.2 (재현율 {tpr * 100:.1f}%, 거짓 양성률 {fpr * 100:.1f}%)의 정밀도")
ax.plot(pi * 100, pi * 100, color=GRAY, lw=0.9, ls=(0, (4, 3)), label="기준율과 같은 정밀도(무작위)")
ax.scatter([y.mean() * 100], [prec * 100], color=BRICK, s=36, zorder=3)
ax.annotate(f"이 자료: 기준율 {y.mean() * 100:.1f}%\n정밀도 {prec * 100:.1f}%", (y.mean() * 100, prec * 100), (22, 12), fontsize=8, color=BRICK, arrowprops={"arrowstyle": "->", "color": BRICK, "lw": 0.8})
ax.scatter([pi2 * 100], [prec2 * 100], color=TEAL, s=36, zorder=3)
ax.annotate(f"폐업을 5분의 1로 줄인 자료:\n기준율 {pi2 * 100:.1f}%, 정밀도 {prec2 * 100:.1f}%", (pi2 * 100, prec2 * 100), (9, 33), fontsize=8, color=TEAL, arrowprops={"arrowstyle": "->", "color": TEAL, "lw": 0.8})
ax.set_xlim(0, 50); ax.set_ylim(0, 62)
ax.set_xlabel("기준율(전체 가운데 폐업하는 가게의 비율, %)", fontsize=8)
ax.set_ylabel("정밀도(%)", fontsize=8)
ax.legend(frameon=False, fontsize=8, loc="lower right")
frame(ax)
fig.tight_layout()
fig.savefig("fig_6_8.png", dpi=300)
plt.show()